# WasteSort primary set (synthetic): generate, check, compare and test

This notebook builds the **generated part** of the study's primary dataset: 2,521 **AI-generated** images of single waste items in campus-like scenes, with the same class counts as the usable TrashNet images (cardboard 403, glass 498, metal 409, paper 594, plastic 480, trash 137). Equal sizes make the primary and secondary datasets directly comparable. The instructor approved synthetic images as the primary dataset. Every image, report and app screen labels them as synthetic. None is a photograph of a real object. The primary dataset's other part, the group's real phone photographs, is already in the repository under `data/primary_set/real_photos/` and is evaluated in the last step.

In Colab choose **Runtime → Change runtime type → GPU (T4 is enough)**, then **Runtime → Run all**. Total time is roughly 2–2.5 hours on a T4. Keep the tab open. Generated candidates are saved to Google Drive, so after a disconnect you can just run all again. All settings are fixed in `protocol_primary.json` before generation. Do not change them after looking at results.

The published TrashNet study (`reports/`, `models/deployment.json`) is not changed. New outputs go to `data/primary_set/`, `reports/primary_set/` and `models/mobilenet_primary.pt`.

In [ ]:
# Step 1: get the code. Set BRANCH to the branch that contains the primary-set code.
BRANCH = "main"
USE_DRIVE = True  # True keeps generated candidates in Google Drive, so a disconnect does not lose them.
import os, subprocess, sys
from pathlib import Path
source = Path("/content/wastesort")
if not source.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "https://github.com/haerinfan/wastesort.git", str(source)], check=True)
os.chdir(source)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements-generate.txt"], check=True)
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        kept = Path("/content/drive/MyDrive/wastesort_primary_candidates")
        kept.mkdir(parents=True, exist_ok=True)
        link = source / "data/primary_set/candidates"
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            link.symlink_to(kept, target_is_directory=True)
        print("Candidates are kept in Google Drive:", kept)
    except Exception as error:  # Drive is optional: a failed mount must not stop the run.
        print("Google Drive not mounted (", error, "). Continuing with local storage;",
              "a disconnect will restart generation from the beginning.")
def run(module, *args):
    subprocess.run([sys.executable, "-u", "-m", "wastesort." + module, *args], check=True)

## Step 2: generate candidates (GPU, about 60–90 minutes)
SDXL-Turbo makes about 4,540 candidates (1.8 × each class target) from the fixed prompt plan. CLIP then scores each image against the six class descriptions. Re-running this cell resumes where it stopped.

In [ ]:
run("generate_primary")

## Step 3: quality control, deduplication and grouped split
Images CLIP does not recognise as their class are rejected, along with exact and near duplicates. The kept images are selected to the TrashNet class counts and split 70/15/15 by item and setting group. If a class has too few images, the cell stops: raise `generator.candidate_multiplier` in `protocol_primary.json` **and record why**, then rerun steps 2–3 (only new candidates are generated). Do not loosen the quality rule.

In [ ]:
run("primary_set")
import json, pandas as pd
from IPython.display import display, Image as Show
display(pd.read_csv("reports/primary_set/split_counts.csv"))
print(json.dumps(json.load(open("reports/primary_set/audit.json")), indent=2))
print("Open reports/primary_set/contact_sheet.html from the file browser to review every image.")

## Step 4: download the secondary TrashNet images
The download uses the same pinned archive and checksum as the original study. Nothing is retrained.

In [ ]:
subprocess.run([sys.executable, "-c", "from wastesort.data import download; download()"], check=True)

## Step 5: compare primary and secondary dataset statistics

In [ ]:
run("dataset_compare")
display(pd.read_csv("reports/primary_set/dataset_statistics.csv"))
for name in ["class_distribution_comparison", "image_statistics_comparison", "embedding_pca", "sample_grid"]:
    display(Show(filename=f"reports/primary_set/{name}.png"))

## Step 6: experiments
**A. Domain shift:** the frozen TrashNet models are evaluated, unchanged, on the primary set.
**B. Fine-tuning:** MobileNetV2 is trained on TrashNet + primary-set training images. Its epoch is chosen on validation only, then it is tested once on both test splits.

In [ ]:
run("primary_experiments", "--stage", "all")
display(pd.read_csv("reports/primary_set/domain_shift.csv"))
display(pd.read_csv("reports/primary_set/finetune_comparison.csv"))
display(Show(filename="reports/primary_set/mobilenet_domain_shift_confusion.png"))

## Step 7: download the results and add them to the repository
Unzip the archive at the repository root. It adds `data/primary_set/`, `reports/primary_set/` and the fine-tuned model. Then run the tests and commit. Raw rejected candidates are not included.

In [ ]:
import shutil, zipfile
archive = Path("/content/WasteSort_primary_set.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["data/primary_set", "reports/primary_set"]:
        for path in Path(folder).rglob("*"):
            if path.is_file() and "candidates" not in path.relative_to(folder).parts:
                z.write(path, path.as_posix())
    for name in ["models/mobilenet_primary.pt", "models/primary_finetuned.json", "protocol_primary.json"]:
        if Path(name).exists():
            z.write(name, name)
print(archive, round(archive.stat().st_size / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Download the ZIP from the file browser.")

## Group review
- Open the contact sheet and record any wrong or unclear image in a review note. Do not silently edit labels.
- Report the primary set as **AI-generated synthetic data** in the manuscript, with the generator, the prompt plan and the QC rejection counts.
- Results on synthetic scenes measure transfer to generated images, not accuracy on real campus waste.

Generator: [SDXL-Turbo](https://huggingface.co/stabilityai/sdxl-turbo) (Stability AI non-commercial research community license). QC: [CLIP ViT-B/32](https://huggingface.co/openai/clip-vit-base-patch32). AI assisted the code; the group verifies and explains it.